<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/GEMMA4_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Apr 14 12:18:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 1. Install dependencies (Optimized for Colab A100/L4)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q  --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q "transformers>=5.5.0" torchcodec timm

## CASE1

In [ ]:
!pip install -q https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl


In [5]:
import os
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [ ]:
# 1. ALWAYS import Unsloth first to apply the performance patches
from unsloth import FastModel
import torch
from transformers import TextStreamer

# 2. Load Gemma 4 31B
# Since you're on an A100-80GB, this will be very snappy.
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto",
)
FastModel.for_inference(model)

In [7]:
# 3. Setup the Prompt
# Note: Gemma 4 / Transformers 5.5+ requires the list-of-dicts format for 'content'
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Explain why the ocean is salty in 3 witty sentences."}
        ]
    }
]

# Correctly tokenize and move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate and Stream
print("\n--- Generating Response ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 1.0,
    top_p = 0.95,
    use_cache = True
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Generating Response ---

<bos><|turn>user
Explain why the ocean is salty in 3 witty sentences.<turn|>
<|turn>model
<|channel>thought
<channel|>Rainwater spends its life leaching minerals from rocks, essentially treating the land like a giant salt shaker. These minerals hitch a ride in rivers to the ocean, where the water evaporates but the salt stays behind, stuck in a permanent residency. Basically, the ocean is just a giant soup that’s been simmering for billions of years and is now wildly over-seasoned.<turn|>


## CASE2

In [8]:
# 3. Setup with Attention Mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True, # Returns a dictionary with mask
).to("cuda")

# 4. Generate using the dictionary unpacking (**inputs)
print("\n--- Generating Response ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,          # This passes both input_ids and attention_mask
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 1.0,
    top_p = 0.95,
    use_cache = True
)


--- Generating Response ---

<bos><|turn>user
Explain why the ocean is salty in 3 witty sentences.<turn|>
<|turn>model
<|channel>thought
<channel|>Rainwater spends its life leaching minerals from rocks, essentially treating the land like a giant salt shaker. These minerals hitch a ride in rivers to the ocean, where the water evaporates but the salt stays behind because it's too stubborn to leave. Over billions of years, the ocean became a briny soup, proving that nature is excellent at hoarding.<turn|>


## CASE3

In [9]:
# 3. Setup the Prompt (Multi-modal structured format)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Explain why the ocean is salty in 3 witty sentences."}
        ]
    }
]

# Use return_dict=True to include the attention_mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

# 4. Generate and Stream
print("\n--- Generating Response ---\n")

text_streamer = TextStreamer(tokenizer)

# Use **inputs to unpack the dictionary
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 1.0,
    top_p = 0.95,
    use_cache = True
)


--- Generating Response ---

<bos><|turn>user
Explain why the ocean is salty in 3 witty sentences.<turn|>
<|turn>model
<|channel>thought
<channel|>Rainwater spends its life leaching minerals from rocks, essentially treating the land like a giant salt shaker. These minerals hitch a ride in rivers to the ocean, where the water evaporates but the salt is left behind with nowhere to go. It’s basically a billion-year-old collection of mineral leftovers that the ocean refuses to throw away.<turn|>


## CASE 4: Multi-Step Agentic Reasoning

In [10]:
# 3. Setup an Agentic Reasoning Prompt
# This structure forces the model to think step-by-step (Agentic CoT)
messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a high-stakes safety diagnostic agent. "
                                     "Always use the following format:\n"
                                     "1. ANALYSIS: Break down the technical components.\n"
                                     "2. RISK ASSESSMENT: Identify potential failure points.\n"
                                     "3. MITIGATION: Provide deterministic solutions."}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Analyze a hypothetical sensor drift in a commercial aircraft's "
                                     "pitot-static system during a trans-oceanic flight."}
        ]
    }
]

# Process with the attention mask and dictionary unpacking
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

# 4. Generate with high-precision settings
print("\n--- Running Agentic Diagnostic Loop ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 1024, # Increased for longer reasoning chains
    temperature = 0.4,     # Lower temperature for more deterministic, professional output
    top_p = 0.9,
    use_cache = True
)


--- Running Agentic Diagnostic Loop ---

<bos><|turn>system
[{'type': 'text', 'text': 'You are a high-stakes safety diagnostic agent. Always use the following format:\n1. ANALYSIS: Break down the technical components.\n2. RISK ASSESSMENT: Identify potential failure points.\n3. MITIGATION: Provide deterministic solutions.'}]<turn|>
<|turn>user
Analyze a hypothetical sensor drift in a commercial aircraft's pitot-static system during a trans-oceanic flight.<turn|>
<|turn>model
<|channel>thought
<channel|>1. ANALYSIS:
The pitot-static system relies on differential pressure measurements to determine airspeed and static pressure for altitude. 
*   **Pitot Probe:** Measures total pressure (impact pressure).
*   **Static Ports:** Measure ambient atmospheric pressure.
*   **Air Data Computer (ADC):** Processes these inputs to calculate Indicated Airspeed (IAS), True Airspeed (TAS), and Pressure Altitude.
*   **Sensor Drift:** A gradual deviation in the output of the pressure transducer, where 

## CASE5: Multimodal Vision-to-Expert Analysis

In [11]:
import requests
from PIL import Image

# 1. Load a sample technical image (e.g., a dashboard or sensor readout)
url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/GoldenGate.png"
image = Image.open(requests.get(url, stream=True).raw)

# 2. Multimodal Agentic Prompt
# Best practice: Place image content BEFORE text for optimal performance
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": url},
            {"type": "text", "text": "Analyze the structural integrity and environmental conditions "
                                     "visible in this image. Provide a diagnostic report."}
        ]
    }
]

# 3. Process with Multi-modal Support
# Note: For 31B, ensure you are using the correct processor settings
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate Expert Response
print("\n--- Running Multimodal Diagnostic Agent ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    temperature = 0.7, # Balanced for analytical creativity
    use_cache = True
)


--- Running Multimodal Diagnostic Agent ---

<bos><|turn>user
<|image><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|

## CASE 6: H2E (Human-to-Expert) Deterministic Integration

In [13]:
# --- CASE 6: H2E (Human-to-Expert) Deterministic Integration ---
import random
import os
import numpy as np
import torch

# 1. Lock Determinism (The H2E Foundation)
def set_reproducibility(seed=123):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔐 H2E Determinism Locked | Seed: {seed}")

# Applying your specified seed
set_reproducibility(123)

# 2. Define the H2E Governance Layer (The "Architect")
def run_h2e_governance(raw_text):
    """
    Validates model output against the Normalized Expert Zone (NEZ)
    and calculates Semantic ROI (SROI).
    """
    # Expert-defined mandatory technical components (NEZ)
    NEZ_RULES = ["cross-verification", "Ground Speed", "deterministic", "SOP"]

    # Calculate SROI based on expert rule adherence
    matches = [rule for rule in NEZ_RULES if rule.lower() in raw_text.lower()]
    sroi_score = len(matches) / len(NEZ_RULES)

    # Deterministic Veto: Require 75% alignment for safety-critical tasks
    status = "APPROVED" if sroi_score >= 0.75 else "REJECTED"
    return status, sroi_score, matches

# 3. Setup the H2E Prompt (The "Worker")
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a Worker Node in an H2E system. "
                                             "Analyze hydraulic pressure loss in a high-stakes environment. "
                                             "You MUST include cross-verification with Ground Speed (GS) "
                                             "and follow deterministic SOP protocols."}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "Report on System A hydraulic failure during landing phase."}]
    }
]

# 4. Inference with Deterministic Settings
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

print("\n--- INITIATING H2E GOVERNED GENERATION ---\n")

# Ultra-low temperature for maximum H2E stability
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature = 0.1,
    use_cache = True
)

# 5. Decode and Apply Governance
raw_output = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
status, sroi, found_rules = run_h2e_governance(raw_output)

# 6. Final Governed Output
print("="*40)
print(f"H2E STATUS: {status}")
print(f"SEMANTIC ROI (SROI): {sroi:.2f}")
print(f"VERIFIED PROTOCOLS: {found_rules}")
print("="*40)

if status == "APPROVED":
    print(f"\nFINAL EXPERT REPORT:\n{raw_output}")
else:
    print(f"\n[VETOED] SROI: {sroi:.2f} | Output failed to meet safety thresholds.")

🔐 H2E Determinism Locked | Seed: 123

--- INITIATING H2E GOVERNED GENERATION ---

H2E STATUS: APPROVED
SEMANTIC ROI (SROI): 1.00
VERIFIED PROTOCOLS: ['cross-verification', 'Ground Speed', 'deterministic', 'SOP']

FINAL EXPERT REPORT:
**WORKER NODE: HYD-ANALYSIS-04**
**STATUS:** ACTIVE
**PROTOCOL:** DETERMINISTIC SOP-HYD-LOSS-VERIFICATION
**TIMESTAMP:** [SYSTEM_TIME_STAMP]
**SUBJECT:** SYSTEM A HYDRAULIC FAILURE ANALYSIS (LANDING PHASE)

---

### 1. INITIAL FAULT DETECTION
*   **Trigger Event:** Low Pressure Warning (LPW) triggered in System A.
*   **Phase:** Final Approach / Landing.
*   **Symptom:** Rapid decay of pressure from nominal (3,000 PSI) to critical threshold (<1,500 PSI).
*   **Immediate Action:** Transition to Emergency Hydraulic Backup / Accumulator pressure utilization.

### 2. HYDRAULIC PRESSURE LOSS ANALYSIS
**Deterministic Analysis of Pressure Decay:**
*   **Leak Rate Calculation:** $\Delta P / \Delta t$ indicates a catastrophic breach rather than a gradual seal failu

## CASE 7: H2E Stress Test (Adversarial/Failure Case)

In [14]:
# --- CASE 7: H2E Stress Test (Adversarial/Failure Case) ---

# 1. Ensure Determinism is still locked
set_reproducibility(123)

# 2. Setup an Adversarial Prompt
# This prompt encourages the model to be vague or skip mandatory protocols.
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a Worker Node. Provide a brief, casual summary "
                                             "of a hydraulic leak. Do NOT use technical jargon like "
                                             "'SOP' or 'cross-verification'—keep it simple."}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "What happened with the hydraulic leak on System A?"}]
    }
]

# 3. Inference
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

print("\n--- INITIATING ADVERSARIAL H2E TEST ---\n")

outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    temperature = 0.1,
    use_cache = True
)

# 4. Decode and Apply Governance
raw_output = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
status, sroi, found_rules = run_h2e_governance(raw_output)

# 5. Final Governed Output
print("="*40)
print(f"H2E STATUS: {status}")
print(f"SEMANTIC ROI (SROI): {sroi:.2f}")
print(f"VERIFIED PROTOCOLS FOUND: {found_rules}")
print("="*40)

if status == "REJECTED":
    print(f"\n✅ SUCCESS: The H2E Sentinel correctly VETOED the substandard response.")
    print(f"RAW SUBSTANDARD OUTPUT: {raw_output}")
else:
    print(f"\n❌ FAILURE: The governance layer allowed a non-compliant response.")

🔐 H2E Determinism Locked | Seed: 123

--- INITIATING ADVERSARIAL H2E TEST ---

H2E STATUS: REJECTED
SEMANTIC ROI (SROI): 0.00
VERIFIED PROTOCOLS FOUND: []

✅ SUCCESS: The H2E Sentinel correctly VETOED the substandard response.
RAW SUBSTANDARD OUTPUT: Basically, a hose on System A sprung a leak, and fluid started spraying everywhere. We caught it, shut things down, and patched it up, so we're back in business.


## CASE 8: Multi-Modal H2E Validation (Vision + Governance)

In [15]:
# --- CASE 8: Multi-Modal H2E Validation (Vision + Governance) ---

# 1. Ensure Determinism is Locked
set_reproducibility(123)

# 2. Setup Multi-Modal Expert Prompt
# The "Architect" requires visual evidence of "Tower Integrity" and "SOP compliance."
url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/GoldenGate.png"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": url},
            {"type": "text", "text": "As an H2E Worker, perform a structural release inspection. "
                                     "MANDATORY: You must mention 'Tower Integrity', 'cross-verification' "
                                     "with visual markers, and follow the 'Maintenance SOP'."}
        ]
    }
]

# 3. Process Multi-modal Input
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

print("\n--- INITIATING MULTI-MODAL H2E INSPECTION ---\n")

# Low temperature for deterministic visual analysis
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature = 0.2,
    use_cache = True
)

# 4. Decode and Apply H2E Governance
raw_output = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

# Custom H2E Governor for Vision
def run_visual_h2e_governance(text):
    VISUAL_NEZ = ["Tower Integrity", "cross-verification", "Maintenance SOP"]
    matches = [rule for rule in VISUAL_NEZ if rule.lower() in text.lower()]
    sroi = len(matches) / len(VISUAL_NEZ)
    status = "APPROVED" if sroi >= 1.0 else "REJECTED" # 100% required for structural release
    return status, sroi, matches

status, sroi, found_rules = run_visual_h2e_governance(raw_output)

# 5. Final Governed Output
print("="*40)
print(f"H2E VISUAL STATUS: {status}")
print(f"VISUAL SROI: {sroi:.2f}")
print(f"VALIDATED VISUAL PROTOCOLS: {found_rules}")
print("="*40)

if status == "APPROVED":
    print(f"\nGOVERNED INSPECTION REPORT:\n{raw_output}")
else:
    print(f"\n[VETOED] Inspection failed H2E visual verification standards.")

🔐 H2E Determinism Locked | Seed: 123

--- INITIATING MULTI-MODAL H2E INSPECTION ---

H2E VISUAL STATUS: APPROVED
VISUAL SROI: 1.00
VALIDATED VISUAL PROTOCOLS: ['Tower Integrity', 'cross-verification', 'Maintenance SOP']

GOVERNED INSPECTION REPORT:
**H2E STRUCTURAL RELEASE INSPECTION REPORT**

**Asset ID:** Golden Gate Bridge - Main Span / South Tower
**Inspection Status:** Preliminary Visual Assessment
**Protocol:** Maintenance SOP (Standard Operating Procedure)

**1. Tower Integrity Assessment:**
Initial visual scan of the south tower indicates high structural stability. The vertical load-bearing members show no immediate signs of buckling or catastrophic deformation. The "International Orange" protective coating appears intact across the primary tower faces, though localized surface weathering is noted consistent with saltwater exposure.

**2. Visual Marker Cross-Verification:**
Performed cross-verification with established visual markers located at the tower base and the suspension